In [3]:
pip install yfinance

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 18.0 MB/s  0:00:00

   ---------------------------------------- 0/5 [peewee]
   ---------------- ----------------------- 2/5 [websockets]
   ------------------------ --------------- 3/5 [curl_cffi]
   -------------------------------- ------- 4/5 [yfinance]
   ---------------------------------------- 5/5 [yfinance]

Note: you may need to restart the kernel to use updated packages.


In [113]:
import yfinance as yf
import numpy as np
import pandas as pd
import talib

In [114]:
# import data from ticker 

ydf = yf.download("NVDA",
                  start="2015-01-01",
                  end="2026-04-30",
                  progress=False)

In [115]:
print(f"Downloaded {len(ydf)} rows of data.")

Downloaded 2847 rows of data.


In [116]:
ydf

Price,Close,High,Low,Open,Volume
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA
Date,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000
2015-01-06,0.459895,0.475473,0.459416,0.474994,197764000
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000
...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400


In [117]:
ydf.columns

MultiIndex([( 'Close', 'NVDA'),
            (  'High', 'NVDA'),
            (   'Low', 'NVDA'),
            (  'Open', 'NVDA'),
            ('Volume', 'NVDA')],
           names=['Price', 'Ticker'])

In [118]:
# Create column for the length of the candle body and daily price action (Total candle range)
ydf['Candle_Body'] = (ydf[('Close', 'NVDA')] - ydf[('Open', 'NVDA')]).abs()
ydf['Total price action'] = (ydf[('High', 'NVDA')] - ydf[('Low', 'NVDA')])

# Create a column to determine if Candle is bullish, bearish or a doji
doji = abs(ydf['Close', 'NVDA'] - ydf['Open', 'NVDA']) <= (0.10 * (ydf['High', 'NVDA'] - ydf['Close', 'NVDA']))
ydf['Bullish/Bearish'] = np.where(
    doji,
    'Doji',
    np.where((ydf['Close', 'NVDA'] > ydf['Open', 'NVDA']), 'Bullish', 'Bearish')
)

# Create column for candle body percentage by dividing the candle body by the total price action of the day
ydf['Body %'] = (ydf['Candle_Body'] / ydf['Total price action']) * 100

# Change total price action column to Candle_range 

ydf = ydf.rename(columns={'Total price action': 'Candle_Range'})

ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,
Date,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Doji,0.000000
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,69.387800
2015-01-06,0.459895,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,94.029920
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,45.237879
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,79.364899
...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,42.662739
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,74.596032


In [119]:
ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,
Date,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Doji,0.000000
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,69.387800
2015-01-06,0.459895,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,94.029920
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,45.237879
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,79.364899
...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,42.662739
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,74.596032


In [120]:
#Add 5, 10, 20, 50, and 200 day EMA Columns 

ydf['5_EMA'] = ydf['Close', 'NVDA'].ewm(span=5, adjust=False).mean()
ydf['10_EMA'] = ydf['Close', 'NVDA'].ewm(span=10, adjust=False).mean()
ydf['20_EMA'] = ydf['Close', 'NVDA'].ewm(span=20, adjust=False).mean()
ydf['50_EMA'] = ydf['Close', 'NVDA'].ewm(span=50, adjust=False).mean()
ydf['200_EMA'] = ydf['Close', 'NVDA'].ewm(span=20, adjust=False).mean()

ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,
Date,,,,,,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Doji,0.000000,0.482423,0.482423,0.482423,0.482423,0.482423
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,69.387800,0.479707,0.480942,0.481647,0.482104,0.481647
2015-01-06,0.459895,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,94.029920,0.473103,0.477115,0.479575,0.481233,0.479575
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,45.237879,0.468301,0.473766,0.477587,0.480349,0.477587
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,79.364899,0.470852,0.474164,0.477431,0.480177,0.477431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,42.662739,199.743683,196.414477,190.964749,186.173520,190.964749
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,74.596032,202.504969,198.525943,192.589777,187.030540,192.589777


In [121]:
pip install TA-lib

Note: you may need to restart the kernel to use updated packages.


In [122]:
import talib

In [123]:
ydf['RSI'] = talib.RSI(ydf['Close', 'NVDA'])
ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA,RSI
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,,
Date,,,,,,,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Doji,0.000000,0.482423,0.482423,0.482423,0.482423,0.482423,NaN
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,69.387800,0.479707,0.480942,0.481647,0.482104,0.481647,NaN
2015-01-06,0.459895,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,94.029920,0.473103,0.477115,0.479575,0.481233,0.479575,NaN
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,45.237879,0.468301,0.473766,0.477587,0.480349,0.477587,NaN
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,79.364899,0.470852,0.474164,0.477431,0.480177,0.477431,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,42.662739,199.743683,196.414477,190.964749,186.173520,190.964749,64.645029
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,74.596032,202.504969,198.525943,192.589777,187.030540,192.589777,71.498286


In [133]:
#Calculate the upper wick by subtracking the high from either the open or close. Use max function and axis=1 to choose the option closest to the upperwick
ydf['Upper_wick'] = abs(ydf['High', 'NVDA'] - ydf[[('Open', 'NVDA'), ('Close', 'NVDA')]].max(axis=1))
# Caculate the same for the lower wick using the min function with axis=1 so it chooses the value closest to the lower wick between the open and close
ydf['Lower_wick'] = abs(ydf['Low', 'NVDA'] - ydf[[('Open', 'NVDA'), ('Close', 'NVDA')]].min(axis=1))

# Calculate wick percentage by dividing the absolute value of the upper or lower wick by the candle ranage (total candle length)
ydf['Upper_wick_percentage'] = abs(ydf['Upper_wick'] / ydf['Candle_Range']) * 100
ydf['Lower_wick_percentage'] = abs(ydf['Lower_wick'] / ydf['Candle_Range']) * 100

# Calculate Wick to body ratio by dividing the sum of the wicks by the candle body
ydf['Wick_to_body_ratio'] = (ydf['Lower_wick'] + ydf['Upper_wick']) / ydf['Candle_Body']

# Calsulate the gap size 
ydf['gap_size'] = ydf['Open', 'NVDA'] - ydf['Close', 'NVDA'].shift(1)
# Calculate gap percentage 
ydf['gap_percentage'] = (ydf['gap_size'] / ydf['Close', 'NVDA'].shift(1)) * 

ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,...,50_EMA,200_EMA,RSI,Upper_wick,Lower_wick,Upper_wick_percentage,Lower_wick_percentage,Wick_to_body_ratio,gap_size,gap_percentage
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,...,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Doji,0.000000,0.482423,...,0.482423,0.482423,NaN,0.003595,0.007669,31.915002,68.084998,inf,NaN,NaN
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,69.387800,0.479707,...,0.482104,0.481647,NaN,0.001438,0.002157,12.244977,18.367223,0.441176,-4.504518e-08,-9.337276e-08
2015-01-06,0.459895,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,94.029920,0.473103,...,0.481233,0.479575,NaN,0.000479,0.000479,2.985040,2.985040,0.063491,7.188530e-04,1.515689e-03
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,45.237879,0.468301,...,0.480349,0.477587,NaN,0.004074,0.001438,40.476326,14.285795,1.210537,3.355263e-03,7.295708e-03
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,79.364899,0.470852,...,0.480177,0.477431,NaN,0.002876,0.000240,19.047727,1.587374,0.260003,5.272332e-03,1.149414e-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,42.662739,199.743683,...,186.173520,190.964749,64.645029,1.368400,2.417181,20.726097,36.611164,1.343966,-3.994972e-02,-1.975125e-04
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,74.596032,202.504969,...,187.030540,192.589777,71.498286,2.676873,0.149835,24.057386,1.346581,0.340554,3.196261e-01,1.602878e-03


In [130]:
ydf.iloc[1500:1550]

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA,RSI,Upper_wick,Lower_wick,Upper_wick_percentage,Lower_wick_percentage
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,
2020-12-16,13.189279,13.383246,13.159398,13.368555,223116000,0.179276,0.223847,Bearish,80.088514,13.189369,13.200899,13.232207,13.142336,13.232207,48.964616,0.014691,0.029880,6.563046,13.348439
2020-12-17,13.287631,13.325727,13.125535,13.313028,231384000,0.025397,0.200192,Bearish,12.686227,13.222123,13.216668,13.237486,13.148034,13.237486,50.906178,0.012699,0.162096,6.343588,80.970185
2020-12-18,13.218658,13.351123,13.017968,13.325228,342064000,0.106569,0.333155,Bearish,31.987923,13.220968,13.217030,13.235693,13.150803,13.235693,49.484424,0.025896,0.200690,7.772817,60.239260
2020-12-21,13.278665,13.316263,12.915381,13.022200,302332000,0.256465,0.400882,Bullish,63.975055,13.240200,13.228236,13.239785,13.155817,13.239785,50.772569,0.037599,0.106819,9.378939,26.646006
2020-12-22,13.224883,13.283646,13.001535,13.265719,185580000,0.040836,0.282111,Bearish,14.474982,13.235094,13.227627,13.238366,13.158526,13.238366,49.552931,0.017927,0.223349,6.354690,79.170328
2020-12-23,12.956964,13.221645,12.952731,13.202722,179144000,0.245758,0.268914,Bearish,91.389012,13.142384,13.178415,13.211566,13.150621,13.211566,43.895977,0.018924,0.004232,7.037070,1.573919
2020-12-24,12.941527,13.079720,12.886749,12.984852,97884000,0.043325,0.192971,Bearish,22.451652,13.075432,13.135345,13.185848,13.142422,13.185848,43.587237,0.094867,0.054778,49.161502,28.386846
2020-12-28,12.848153,13.010000,12.711953,13.010000,212564000,0.161847,0.298047,Bearish,54.302614,12.999672,13.083128,13.153686,13.130882,13.153686,41.677634,0.000000,0.136200,0.000000,45.697386


In [98]:
this can be used to compare columns 
ydf.shift()

SyntaxError: invalid syntax (135002856.py, line 1)

In [10]:
ydf['Low']

Ticker,NVDA
Date,
2015-01-02,0.474754
2015-01-05,0.472118
2015-01-06,0.459416
2015-01-07,0.457259
2015-01-08,0.463730
...,...
2025-12-23,182.677196
2025-12-24,186.362690
2025-12-26,187.770981


In [11]:
pip install nasdaq-data-link

In [36]:
import nasdaqdatalink as ndl

In [37]:
My_Nas_Key = "bp9V2Fn_MCJ8xEUxYS8p"

ndl.ApiConfig.api_key = My_Nas_Key

In [80]:
ndf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_range (price action),Bullish/Bearish,Total price action,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA,RSI
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,,,
Date,,,,,,,,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Bearish,0.011264,0.000000,0.482423,0.482423,0.482423,0.482423,0.482423,NaN
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,0.011743,69.387800,0.479707,0.480942,0.481647,0.482104,0.481647,NaN
2015-01-06,0.459895,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,0.016057,94.029920,0.473103,0.477115,0.479575,0.481233,0.479575,NaN
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,0.010065,45.237879,0.468301,0.473766,0.477587,0.480349,0.477587,NaN
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,0.015098,79.364899,0.470852,0.474164,0.477431,0.480177,0.477431,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,6.602306,42.662739,199.743683,196.414477,190.964749,186.173520,190.964749,64.645029
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,11.127031,74.596032,202.504969,198.525943,192.589777,187.030540,192.589777,71.498286
